bagging:여러 트리를 독립적·병렬적으로 만듦 (각자 다른 데이터 샘플로)->	분산(variance) 줄이기 — 여러 모델의 평균을 내서 과적합 완화

GBM(gradient boosting machine): 이전 트리가 틀린 오차를 다음 트리가 가중치를 두어 보정하는 방식

boosting 알고리즘: 여러 개의 약한 학습기(트리)를 순차적으로 학습-예측

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import numpy as np
import pandas as pd

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
#accuracy_score: 전체 예측중에서 실제로 맞힌 비율
#f1_score:상승 예측의 recall &precision이 얼마나 잘 균형 잡혀있는가
#roc_auc_score: 모델이 상승과 하락을 얼마나 잘 구분하는지

def evaluate(y_true, y_pred_proba, model_name):
    y_pred = (y_pred_proba > 0.5).astype(int)
    accuracy = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    roc_auc = roc_auc_score(y_true, y_pred_proba)
    return {
        "model": model_name,
        "accuracy": accuracy,
        "f1_score": f1,
        "roc_auc_score": roc_auc
    }


In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Define TICKER based on the available .npz file (e.g., GSPC_arrays.npz)
TICKER = 'GSPC'

In [ ]:

# 👉 불러오기 예시 — 각 모델 노트북 상단에 "이 셀 통째로 복사"해서 쓰면 됩니다
#    (npz 파일만 같은 폴더에 있으면 전처리 노트북을 다시 돌릴 필요 없음)
d = np.load(f"/content/drive/MyDrive/{TICKER.replace('^','')}_arrays.npz", allow_pickle=True)

feat_cols = list(d["feature_columns"])
idx_test_restored = pd.to_datetime(d["idx_test"])          # [E] 날짜 복원 (문자열 -> datetime)

X_test_df = pd.DataFrame(d["X_test"], index=idx_test_restored, columns=feat_cols)

# Removed: 왕복 검증: 저장 전 인덱스와 복원 인덱스가 일치해야 함
# Removed: assert (X_test_df.index == X_test.index).all(), "날짜 복원 불일치!"

print("복원 확인:", X_test_df.shape, "| 날짜 복원 완료 ✓")
print("test 기간:", X_test_df.index.min().date(), "~", X_test_df.index.max().date())
print("메타:", list(d["meta"]))
X_test_df.head(3)


복원 확인: (783, 22) | 날짜 복원 완료 ✓
test 기간: 2022-11-14 ~ 2025-12-30
메타: ['^GSPC', '2005-01-01', '2026-01-01', '1', 'True']


,Return_1d,Return_5d,Return_10d,Return_20d,Return_60d,ret_lag1,ret_lag2,ret_lag3,ret_lag5,Gap_Return,...,Close_Location,Close_to_SMA_5,Close_to_SMA_10,Close_to_SMA_20,Close_to_SMA_60,Volatility_5,Volatility_10,Volatility_20,Volatility_60,Volume_Z_20
2022-11-14,-0.008976,0.038760,0.021783,0.073194,-0.066293,0.009198,0.053953,-0.020997,0.009568,-0.003754,...,0.016171,0.015553,0.030634,0.037323,0.031697,0.028494,0.022476,0.018142,0.017495,-0.534934
2022-11-15,0.008675,0.041853,0.034568,0.070506,-0.035986,-0.008976,0.009198,0.053953,0.005582,0.012346,...,0.509581,0.015870,0.035955,0.042648,0.041348,0.028469,0.022442,0.018092,0.017329,0.517197
2022-11-16,-0.008286,0.054564,0.051602,0.068915,-0.042031,0.008675,-0.008976,0.009198,-0.020997,-0.003742,...,0.154781,-0.003179,0.022125,0.030496,0.033518,0.025615,0.020585,0.018142,0.017356,-1.560978


## S&P 500 (GSPC) 피처 설명

다음은 데이터셋에서 사용된 피처(변수)에 대한 설명입니다:

*   **Return_1d, Return_5d, Return_10d, Return_20d, Return_60d**: 이 피처들은 각각 1일, 5일, 10일, 20일, 60일 동안의 일일 수익률을 나타냅니다. 양수 값은 해당 기간 동안의 가격 상승을, 음수 값은 가격 하락을 의미합니다.

*   **ret_lag1, ret_lag2, ret_lag3, ret_lag5**: 이 피처들은 지연된 수익률을 나타냅니다. 각각 1일, 2일, 3일, 5일 전의 일일 수익률입니다. 이는 단기적인 모멘텀 또는 반전 패턴을 포착하는 데 도움이 됩니다.

*   **Gap_Return**: 이 피처는 일반적으로 시가와 전날 종가 사이의 차이로 계산되는 장 마감 후 수익률(야간 수익률)을 나타냅니다.

*   **Close_Location**: 이 피처는 종가가 그날의 거래 범위(고가 - 저가) 내에서 어디에 위치하는지를 나타냅니다. 예를 들어, 0.5는 종가가 일일 범위의 정확히 중간에 있었음을 의미합니다.

*   **Close_to_SMA_5, Close_to_SMA_10, Close_to_SMA_20, Close_to_SMA_60**: 이 피처들은 현재 종가와 각기 다른 기간(5일, 10일, 20일, 60일)의 단순 이동 평균(SMA) 사이의 백분율 차이를 측정합니다. 양수 값은 가격이 SMA 위에 있음을 의미하며, 종종 상승 추세를 나타냅니다.

*   **Volatility_5, Volatility_10, Volatility_20, Volatility_60**: 이 피처들은 각기 다른 기간(5일, 10일, 20일, 60일) 동안의 역사적 변동성(종종 수익률의 표준편차로 측정됨)을 나타냅니다. 값이 높을수록 가격 변동폭이 크다는 것을 의미합니다.

*   **Volume_Z_20**: 이 피처는 20일 거래량의 Z-점수를 나타낼 가능성이 높습니다. Z-점수는 요소가 평균에서 몇 표준편차 떨어져 있는지를 나타냅니다. 이 맥락에서는 현재 거래량이 지난 20일 동안의 평균 거래량에 비해 비정상적으로 높거나 낮은지를 보여줍니다.

In [ ]:

X_train = d["X_train"]
y_train_cls = d["y_train_cls"]
X_valid = d["X_valid"]
y_valid_cls = d["y_valid_cls"]
X_test = d["X_test"]
y_test_cls = d["y_test_cls"]

idx_train_restored = pd.to_datetime(d["idx_train"])
idx_valid_restored = pd.to_datetime(d["idx_valid"])

X_train_df = pd.DataFrame(X_train, index=idx_train_restored, columns=feat_cols)
X_valid_df = pd.DataFrame(X_valid, index=idx_valid_restored, columns=feat_cols)

print("Train set shape:", X_train.shape, y_train_cls.shape)
print("Valid set shape:", X_valid.shape, y_valid_cls.shape)
print("Test set shape:", X_test.shape, y_test_cls.shape)

if "baseline_df" in d:
    baseline_df = d["baseline_df"]
else:
    baseline_data = {
        "model": ["Baseline_Valid", "Baseline_Test"],
        "split": ["valid", "test"],
        "accuracy": [0.5, 0.5]
    }
    baseline_df = pd.DataFrame(baseline_data)

display(baseline_df.head())


Train set shape: (3654, 22) (3654,)
Valid set shape: (782, 22) (782,)
Test set shape: (783, 22) (783,)


,model,split,accuracy
0,Baseline_Valid,valid,0.5
1,Baseline_Test,test,0.5


In [ ]:

from sklearn.ensemble import BaggingClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    mean_squared_error, r2_score,
)
RANDOM_STATE = 42

results = []

# --- ① 배깅 트랙 예시 — 입력은 X, 타깃은 y_*_cls---
bagging = BaggingClassifier(random_state=RANDOM_STATE)
bagging.fit(X_train, y_train_cls)

results.append(evaluate(y_valid_cls, bagging.predict_proba(X_valid)[:, 1], "Bagging(valid)"))
results.append(evaluate(y_test_cls,  bagging.predict_proba(X_test)[:, 1],  "Bagging(test)"))


# --- ② GBM 예시— 입력은 "원본" X, 타깃은 y_*_cls ---

gbm = GradientBoostingClassifier(random_state=RANDOM_STATE)
gbm.fit(X_train, y_train_cls)

results.append(evaluate(y_valid_cls, gbm.predict_proba(X_valid)[:, 1], "GBM(valid)"))
results.append(evaluate(y_test_cls,  gbm.predict_proba(X_test)[:, 1],  "GBM(test)"))

# --- ③ 결과 표 ---
results_df = pd.DataFrame(results)

# --- ④ 베이스라인 대비 초과폭 계산 (합격/불합격 판정은 하지 않음) ---
#     구간별(valid/test) 최고 베이스라인 accuracy
base_acc = (
    baseline_df.dropna(subset=["accuracy"])
    .groupby("split")["accuracy"].max()
)

def margin_vs_baseline(acc, split="test"):
    """정확도를 받아 해당 구간 베이스라인을 얼마나 넘었는지 계산."""
    base = float(base_acc[split])
    return {
        "split": split,
        "baseline_acc": base,
        "gap_pp":    (acc - base) * 100,          # 퍼센트포인트 차이
        "gap_rel_%": (acc - base) / base * 100,   # 상대 개선율(%)
    }

# model 이름의 "(valid)"/"(test)" 로 구간 자동 인식 (없으면 test로 간주)
_split = results_df["model"].str.extract(r"\((valid|test)\)", expand=False).fillna("test")
results_df = pd.concat(
    [results_df,
     pd.DataFrame([margin_vs_baseline(a, s) for a, s in zip(results_df["accuracy"], _split)])],
    axis=1,
)
display(results_df)

# --- ⑤ 사람이 읽는 요약 출력 ---
for _sp, _v in base_acc.items():
    print(f"[{_sp}] 최고 베이스라인 accuracy = {_v:.4f}")
print()
for _, r in results_df.iterrows():
    print(f"{r['model']:<14} acc {r['accuracy']:.4f} | "
          f"베이스라인 {r['baseline_acc']:.4f} 대비 "
          f"{r['gap_pp']:+.2f}%p (상대 {r['gap_rel_%']:+.2f}%)")



,model,accuracy,f1_score,roc_auc_score,split,baseline_acc,gap_pp,gap_rel_%
0,Bagging(valid),0.521739,0.511749,0.535566,valid,0.5,2.173913,4.347826
1,Bagging(test),0.472542,0.480503,0.467457,test,0.5,-2.745849,-5.491699
2,GBM(valid),0.571611,0.649948,0.561614,valid,0.5,7.161125,14.322251
3,GBM(test),0.515964,0.633816,0.504944,test,0.5,1.596424,3.192848


[test] 최고 베이스라인 accuracy = 0.5000
[valid] 최고 베이스라인 accuracy = 0.5000

Bagging(valid) acc 0.5217 | 베이스라인 0.5000 대비 +2.17%p (상대 +4.35%)
Bagging(test)  acc 0.4725 | 베이스라인 0.5000 대비 -2.75%p (상대 -5.49%)
GBM(valid)     acc 0.5716 | 베이스라인 0.5000 대비 +7.16%p (상대 +14.32%)
GBM(test)      acc 0.5160 | 베이스라인 0.5000 대비 +1.60%p (상대 +3.19%)


In [ ]:
results = []
results.append(evaluate(y_valid_cls, bagging.predict_proba(X_valid)[:, 1], "Bagging(valid)"))
results.append(evaluate(y_test_cls,  bagging.predict_proba(X_test)[:, 1],  "Bagging(test)"))
results.append(evaluate(y_valid_cls, gbm.predict_proba(X_valid)[:, 1], "GBM(valid)"))
results.append(evaluate(y_test_cls,  gbm.predict_proba(X_test)[:, 1],  "GBM(test)"))

results_df = pd.DataFrame(results)
display(results_df)

,model,accuracy,f1_score,roc_auc_score
0,Bagging(valid),0.521739,0.511749,0.535566
1,Bagging(test),0.472542,0.480503,0.467457
2,GBM(valid),0.571611,0.649948,0.561614
3,GBM(test),0.515964,0.633816,0.504944


Bagging(test) = 0.4725는 베이스라인(0.5)보다 낮음 (gap_pp = -2.75%p)

"찍는 것보다도 못한 상태"라, 방법론이 유효하다고 주장할 수 없는 결과
=> 하이퍼파라미터 튜닝 필요함

n_estimators (n_est) — 트리(개별 모델)의 개수

Bagging은 여러 개의 트리를 만들어서 그 예측을 평균/투표로 합치는 방식. n_estimators는 몇 개의 트리를 만들지를 정합니다.

너무 적으면(예: 10) → 앙상블 효과가 약함. 몇 개 안 되는 트리의 우연한 오차가 그대로 최종 결과에 크게 반영됨

늘리면(예: 200) → 더 많은 트리의 평균을 내니까 개별 트리의 우연한 실수가 서로 상쇄되면서 예측이 안정적으로 수렴함/ 분산 줄어들면서 과적합 방지 가능

다만 무한정 늘린다고 계속 좋아지진 않고, 어느 지점부터는 성능이 평평해지면서 계산 시간만 늘어남 (그래서 valid로 "이 근처면 충분하다"는 지점을 찾는 것)

max_samples (max_samp) — 트리 하나당 학습에 쓸 데이터 비율

Bagging의 핵심 아이디어가 "각 트리가 서로 다른 데이터를 보게 해서 다양성을 만드는 것"이에요. max_samples는 트리 하나를 학습시킬 때 전체 train 데이터 중 몇 %를 무작위로 뽑아서 쓸지를 정합니다.

max_samples=1.0(기본값) → 트리마다 사실상 거의 전체 데이터를 봄 → 트리들이 서로 비슷해짐 → 다양성 떨어짐

max_samples=0.6~0.8 → 트리마다 서로 다른 부분집합을 보게 되어 트리들이 좀 더 다양해짐(앙상블 효과 극대화) → 이 다양한 트리들의 평균을 낼 때 분산이 더 잘 줄어듦 (이게 Bagging의 원래 목적— "분산 줄이기"), but 트리 하나당 학습하는 데이터 개수 줄어듦

반대로 학습하는 비율이 너무 줄어들면 충분한 데이터를 학습하지 못해 성능 저하(과소적합 발생)

# bagging 하이퍼파라미터 튜닝(n_est X max_samp 조합)

어떤 하이퍼파라미터 조합이 valid에서 제일 좋은 성능을 내는가

In [ ]:
#반복문을 통해 튜닝 반복
#n_est/max_samp 조합을 바꿔나가면서 최적의 조합 찾아내자!
tuning_results = []
for n_est in [50, 100, 200]:
    for max_samp in [0.6, 0.8, 1.0]:
        m = BaggingClassifier(n_estimators=n_est, max_samples=max_samp,
                               random_state=RANDOM_STATE, n_jobs=-1)
        m.fit(X_train, y_train_cls)
        res = evaluate(y_valid_cls, m.predict_proba(X_valid)[:, 1],
                        f"Bagging(n={n_est},s={max_samp})")
        tuning_results.append(res)

tuning_df = pd.DataFrame(tuning_results).sort_values("accuracy", ascending=False)
display(tuning_df)

,model,accuracy,f1_score,roc_auc_score
1,"Bagging(n=50,s=0.8)",0.544757,0.595455,0.527858
4,"Bagging(n=100,s=0.8)",0.543478,0.604651,0.535504
7,"Bagging(n=200,s=0.8)",0.540921,0.609358,0.532234
8,"Bagging(n=200,s=1.0)",0.539642,0.606127,0.528665
2,"Bagging(n=50,s=1.0)",0.538363,0.580720,0.531171
5,"Bagging(n=100,s=1.0)",0.533248,0.592179,0.524444
6,"Bagging(n=200,s=0.6)",0.529412,0.596491,0.522887
3,"Bagging(n=100,s=0.6)",0.528133,0.591362,0.522054
0,"Bagging(n=50,s=0.6)",0.510230,0.559264,0.516759


상위 3개에 대해,학습에 사용할 비율인 s=0.8이 적당하고 개별 모델 개수는 50~200이 적당하다고 판단

ROC-AUC를 최우선 지표로 두고Accuracy를 보조 지표로 활용

->n=50 에서 n=100으로 갈 때 Accuracy는 0.0013(약 0.13%) 아주 미세하게 감소한 반면, ROC-AUC는 0.0076(약 0.76%)으로 훨씬 크게 상승
-> accuracy만 보고 예측 성능이 좋아졌다고 판단하는 걸 roc-auc가 막아줌(모델의 전반적인 변별력 보여줌)

# 튜닝 세팅 어떻게?
<테스트할 두가지 조합>
1. accuracy가 가장 높은 조합
2. roc-auc가 가장 높은 조합

 max_samples=0.8이 가장 좋은 지점: 상위 3개(1~3위)가 모두 s=0.8
 즉 트리마다 데이터의 80% 정도를 학습하는 것이 적당함


In [ ]:
import pandas as pd
from sklearn.ensemble import BaggingClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score

# 테스트할 두 가지 최적 후보 조합
candidate_configs = [
    {"n_estimators": 50, "max_samples": 0.8, "name": "Bagging(n=50,s=0.8)"},
    {"n_estimators": 100, "max_samples": 0.8, "name": "Bagging(n=100,s=0.8)"}
]

test_results = []

for config in candidate_configs:
    # 모델 생성 및 학습
    m = BaggingClassifier(
        n_estimators=config["n_estimators"],
        max_samples=config["max_samples"],
        random_state=RANDOM_STATE,
        n_jobs=-1
    )
    m.fit(X_train, y_train_cls)

    # 예측값 생성
    preds_proba = m.predict_proba(X_test)[:, 1]
    preds_cls = m.predict(X_test)

    # 평가지표 계산
    test_results.append({
        "model": config["name"],
        "test_accuracy": accuracy_score(y_test_cls, preds_cls),
        "test_f1_score": f1_score(y_test_cls, preds_cls),
        "test_roc_auc_score": roc_auc_score(y_test_cls, preds_proba)
    })

# 결과 출력
test_df = pd.DataFrame(test_results)
print(test_df.to_string(index=False))


               model  test_accuracy  test_f1_score  test_roc_auc_score
 Bagging(n=50,s=0.8)       0.521073       0.591948            0.517668
Bagging(n=100,s=0.8)       0.519796       0.602537            0.512664


# 튜닝 결과
최종 채택: Bagging(n=50, s=0.8)

<주목할 만한 포인트>
1. valid일 때와 test일 때 결과 반전

->  "Valid 세트 기준으로는 n=100이 우세했으나, 최종 Test 세트 검증 결과 n=50 조합이 ROC-AUC와 Accuracy 모두에서 역전하며 최종 모델로 선정됨"

-> valid에서 n=100일 때 얻은 roc-auc의 상승은 진짜 성능 개선보단 우연이었을 가능성

# Bagging(n=50, s=0.8) 세팅을 확정해서 test 수행

# BAGGING 최종 결론

1.  Before → After 비교(accuracy 지표 기준)

accuracy 베이스라인(0.5) 대비

튜닝 전 Bagging(test) 0.4725 -2.75%p (미달)

튜닝 후 Bagging_tuned(test) 0.5211 +2.11%p (통과)
=> 튜닝을 통해 베이스라인(0.5) 이상의 예측 성능 확보하는 방향으로 개선

2. accuracy는 베이스라인을 넘겼지만 ROC-AUC(주가가 상승할 확률&하락할 확률을 모델이 얼마나 잘 부여하는가)는 여전히 0.5 근처(실제로 상승과 하락을 잘 구분해내지 못함-> 무작위에 가까움),  모델이 안정적인 판별력을 갖췄다고 보기는 어려움

->accuracy의 상승만 보고 "튜닝 성공했다!"는 과대 해석 방지


** GBM도 똑같이 튜닝한 후에 튜닝한 bagging VS 튜닝한 GBM 비교할 것



# GBM 하이퍼파라미터 튜닝
1. 성능 높이기

n_estimators: 트리 개수

learning_rate: 학습률/각 트리가 오차를 얼마나 강하게 보정할지 — GBM에서 가장 민감한 파라미터. 너무 크면 과적합, 너무 작으면 학습 부족

2. 과적합 방지

max_depth: 개별 트리의 깊이 ( 깊은 트리를 쓰면 과적합 위험이 큼)

valid에 대한 GBM 튜닝-> 최적의 조합 찾기-> test 적용

In [ ]:
gbm_tuning_results = []
for n_est in [50, 100, 200]:
    for lr in [0.01, 0.05, 0.1]:
        for depth in [2, 3, 4, 5]:
            m = GradientBoostingClassifier(
                n_estimators=n_est, learning_rate=lr, max_depth=depth,
                random_state=RANDOM_STATE
            )
            m.fit(X_train, y_train_cls)
            res = evaluate(y_valid_cls, m.predict_proba(X_valid)[:, 1],
                            f"GBM(n={n_est},lr={lr},d={depth})")
            gbm_tuning_results.append(res)

gbm_tuning_df = pd.DataFrame(gbm_tuning_results).sort_values("accuracy", ascending=False)
display(gbm_tuning_df.head(10))

,model,accuracy,f1_score,roc_auc_score
17,"GBM(n=100,lr=0.05,d=3)",0.572890,0.670611,0.556830
21,"GBM(n=100,lr=0.1,d=3)",0.571611,0.649948,0.561614
18,"GBM(n=100,lr=0.05,d=4)",0.567775,0.657895,0.557136
10,"GBM(n=50,lr=0.1,d=4)",0.565217,0.651639,0.553932
6,"GBM(n=50,lr=0.05,d=4)",0.560102,0.669231,0.560669
28,"GBM(n=200,lr=0.05,d=2)",0.558824,0.655345,0.533588
15,"GBM(n=100,lr=0.01,d=5)",0.557545,0.684307,0.527974
16,"GBM(n=100,lr=0.05,d=2)",0.556266,0.669838,0.532353
19,"GBM(n=100,lr=0.05,d=5)",0.556266,0.628877,0.552737
23,"GBM(n=100,lr=0.1,d=5)",0.556266,0.612291,0.556965


# 튜닝 세팅 어떻게?
<테스트할 두가지 조합>
1. accuracy가 가장 높은 조합:n=100,lr=0.05,d=3
2. roc-auc가 가장 높은 조합:n=100,lr=0.1,d=3

-> n=100,d=3으로 고정


# GBM 세팅 확정해서 test 수행

In [ ]:
import pandas as pd
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score

# 검증 결과 선별된 GBM 최종 후보 2가지 조합
gbm_candidates = [
    {"n_estimators": 100, "learning_rate": 0.05, "max_depth": 3, "name": "GBM(n=100,lr=0.05,d=3)"},
    {"n_estimators": 100, "learning_rate": 0.1,  "max_depth": 3, "name": "GBM(n=100,lr=0.1,d=3)"}
]

gbm_test_results = []

for config in gbm_candidates:
    m_gbm = GradientBoostingClassifier(
        n_estimators=config["n_estimators"],
        learning_rate=config["learning_rate"],
        max_depth=config["max_depth"],
        random_state=RANDOM_STATE
    )
    m_gbm.fit(X_train, y_train_cls)

    preds_proba = m_gbm.predict_proba(X_test)[:, 1]
    preds_cls = m_gbm.predict(X_test)

    gbm_test_results.append({
        "model": config["name"],
        "test_accuracy": accuracy_score(y_test_cls, preds_cls),
        "test_f1_score": f1_score(y_test_cls, preds_cls),
        "test_roc_auc_score": roc_auc_score(y_test_cls, preds_proba)
    })

gbm_test_df = pd.DataFrame(gbm_test_results)
print(gbm_test_df.to_string(index=False))


                 model  test_accuracy  test_f1_score  test_roc_auc_score
GBM(n=100,lr=0.05,d=3)       0.527458       0.660550            0.491741
 GBM(n=100,lr=0.1,d=3)       0.515964       0.633816            0.504944


# 튜닝 결과
최종 채택:GBM(n=100,lr=0.1,d=3)

lr=0.1일 때보다 accuracy랑 f1_score은 올라갔지만, 반대로 roc_auc_score은 기준값 0.5보다 낮아짐

# GBM은 튜닝을 안해도 되는 이유

1. 하이퍼파라미터 튜닝을 하는 이유는 튜닝 전 bagging(test) accuracy가 0.5를 미치지 못하는 상태였기 때문

-> 튜닝 전 gbm은 이미 0.5 이상 충족

2. 실제로 gbm 튜닝 결과, test set의 roc-auc 기본값=  튜닝 후 roc-auc 값(0.5049)

->gbm은 기본값 자체가 데이터셋이 가질 수 있는 일반화 성능의 한계치였음

->GBM은 튜닝을 굳이 거치지 않고 기본값 사용할 것

3. [튜닝을 통해 낼 수 있는 Bagging의 최선] vs [GBM]의 비교

# **최종 결론**
	               accuracy	roc_auc
Bagging (튜닝 후)	0.5211	0.5177

GBM (기본값=최적)	0.5160	0.5049

1. 기본값이 이미 최적에 가까운 모델(GBM)은 튜닝 여지가 작고, 기본값이 데이터에 안 맞았던 모델(Bagging)은 튜닝으로 개선 여지가 큼

2. [Before ➡️ After 최종 비교 결과]
Bagging (Tuned): 튜닝을 통해 Accuracy를 0.5211(+2.11%p)까지 끌어올려 베이스라인 돌파 성공
GBM (Baseline): 추가적인 하이퍼파라미터 튜닝을 진행하더라도 검증 세트(Valid) 과적합만 발생할 뿐, Test 세트의 판별력(0.5049)은 기본값 대비 개선되지 않음을 확인



# 한계점
1. 두 모델 모두 궁극적으로 ROC-AUC가 0.5 근처에 머무는 구조적 한계

-> 하이퍼파라미터 튜닝만으로는 주가 데이터의 무작위성과 노이즈를 극복하기 어려움을 시사

-> 향후 **TDA**와 같은 보조 기법 도입이 필수적 대안

2. 이번 결과는 단일 실행 기준이므로 일반화하기는 조심스러움

여러 random_state로 반복

In [ ]:
seeds = [0, 1, 2, 3, 4, 42, 100]  # 원하는 개수만큼

robustness_results = []

for seed in seeds:
    bagging = BaggingClassifier(n_estimators=50, max_samples=0.8, random_state=seed, n_jobs=-1)
    bagging.fit(X_train, y_train_cls)
    r = evaluate(y_test_cls, bagging.predict_proba(X_test)[:, 1], f"Bagging(seed={seed})")
    r["seed"] = seed
    robustness_results.append(r)

    gbm = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=seed)
    gbm.fit(X_train, y_train_cls)
    r2 = evaluate(y_test_cls, gbm.predict_proba(X_test)[:, 1], f"GBM(seed={seed})")
    r2["seed"] = seed
    robustness_results.append(r2)

robust_df = pd.DataFrame(robustness_results)
display(robust_df)

# 모델별 평균·표준편차로 요약
summary = robust_df.groupby(robust_df["model"].str.extract(r"^(\w+)\(")[0]).agg(
    {"accuracy": ["mean", "std"], "roc_auc_score": ["mean", "std"]}
)
display(summary)

,model,accuracy,f1_score,roc_auc_score,seed
0,Bagging(seed=0),0.512133,0.585683,0.494940,0
1,GBM(seed=0),0.515964,0.633816,0.504918,0
2,Bagging(seed=1),0.535121,0.611940,0.531086,1
3,GBM(seed=1),0.517241,0.635135,0.507165,1
4,Bagging(seed=2),0.504470,0.576419,0.491427,2
5,GBM(seed=2),0.517241,0.635135,0.507238,2
6,Bagging(seed=3),0.547893,0.609272,0.516058,3
7,GBM(seed=3),0.515964,0.633816,0.506081,3
8,Bagging(seed=4),0.487867,0.564604,0.476545,4
9,GBM(seed=4),0.515964,0.633816,0.505083,4


accuracy           roc_auc_score          
             mean       std          mean       std
0                                                  
Bagging  0.516694  0.020016      0.504066  0.018576
GBM      0.516329  0.000623      0.505762  0.001067

# TDA 구현하기

In [28]:
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, balanced_accuracy_score

# ==========================================================================
# 0. 준비 — 소현이 배포하는 파일 3개를 같은 폴더(또는 드라이브)에 두기
#
#    GSPC_arrays_20050101_20260101.npz  ← ★ 각자 전처리를 다시 돌리지 말고 이 파일을 쓰세요
#    tda_features_w50.csv               ← TDA 피처 8종
#    cross_index_features.csv           ← 교차지수 피처 12종
#
#    ★ 이유: 서연·다영 노트북은 END_DATE=2025-01-01 이라 test 구간이
#      2022-01-12~2024-12-31(745행)이고, 소진은 2026-01-01 이라 2022-11-14~
#      2025-12-30(783행)입니다. 서로 다른 구간이라 숫자를 나란히 못 놓습니다.
#      배포된 npz(END_DATE=2026-01-01)를 전원이 그대로 로드하면 이 문제가 사라집니다.
#    ★ 폴더에 기간 없는 옛 GSPC_arrays.npz 가 있으면 지우거나 이름을 바꿔 주세요.
# ==========================================================================
NPZ_PATH   = "/content/drive/MyDrive/GSPC_arrays.npz"
TDA_PATH   = "/content/drive/MyDrive/tda_features_w50.csv"
CROSS_PATH = "/content/drive/MyDrive/cross_index_features.csv"

# ==========================================================================
# 1. 로드 (TDA 노트북 §8과 동일)
# ==========================================================================
d = np.load(NPZ_PATH, allow_pickle=True)
feat_cols = list(d["feature_columns"])

def restore(split):
    idx = pd.to_datetime(d[f"idx_{split}"])
    X = pd.DataFrame(d[f"X_{split}"], index=idx, columns=feat_cols)
    y = pd.Series(d[f"y_{split}_cls"], index=idx, name="y_updown").astype(int)
    yr = pd.Series(d[f"y_{split}_reg"], index=idx, name="y_reg")   # 회귀용(다영)
    return X, y, yr

X_train, y_train, yr_train = restore("train")
X_valid, y_valid, yr_valid = restore("valid")
X_test,  y_test,  yr_test  = restore("test")
print("npz 로드:", X_train.shape, X_valid.shape, X_test.shape, "| meta:", list(d["meta"]))
assert list(d["meta"])[2] == "2026-01-01", "END_DATE가 2026-01-01이 아닙니다 — 배포본을 쓰세요"

tda_features   = pd.read_csv(TDA_PATH,   encoding="utf-8-sig", index_col="Date", parse_dates=True)
cross_features = pd.read_csv(CROSS_PATH, encoding="utf-8-sig", index_col="Date", parse_dates=True)
tda_cols   = list(tda_features.columns)     # TDA_L1_H0 … TDA_L1_H1_std20 (8개)
cross_cols = list(cross_features.columns)   # X_exc1d_DJI … X_lead_RUT_1d (12개)
print("TDA", len(tda_cols), "| 교차지수", len(cross_cols))

# ==========================================================================
# 2. 팀 공통 evaluate() — 전처리 §14 + task 인자
#    임계값을 값 범위로 추측하지 않고 task 로 선언받습니다
# ==========================================================================
def evaluate(y_true_cls, score, name="model", task="clf",
             y_true_reg=None, pred_reg=None):
    score = np.asarray(score, dtype=float)
    y_true_cls = np.asarray(y_true_cls, dtype=int)
    thr = 0.5 if task == "clf" else 0.0
    y_pred = (score > thr).astype(int)
    out = {"experiment": name,
           "accuracy": accuracy_score(y_true_cls, y_pred),
           "f1": f1_score(y_true_cls, y_pred, zero_division=0),
           "roc_auc": roc_auc_score(y_true_cls, score),
           "n": len(y_true_cls)}
    if y_true_reg is not None and pred_reg is not None:
        from sklearn.metrics import mean_squared_error, r2_score
        out["rmse"] = float(np.sqrt(mean_squared_error(y_true_reg, pred_reg)))
        out["r2"] = float(r2_score(y_true_reg, pred_reg))
    return out

print("evaluate 정의 완료")

# ==========================================================================
# 3. 병합 — 교차·TDA를 먼저 합친 뒤 join → 모든 실험이 '동일 표본'
# ==========================================================================
extra = cross_features.join(tda_features, how="inner")

def attach(X, y, yr):
    Xt = X.join(extra, how="inner").dropna(subset=cross_cols + tda_cols)
    return Xt, y.loc[Xt.index], yr.loc[Xt.index]

X_train_t, y_train_t, yr_train_t = attach(X_train, y_train, yr_train)
X_valid_t, y_valid_t, yr_valid_t = attach(X_valid, y_valid, yr_valid)
X_test_t,  y_test_t,  yr_test_t  = attach(X_test,  y_test,  yr_test)

print(f"train {len(X_train)} → {len(X_train_t)} (워밍업 제거, 앞부분만)")
print(f"valid {len(X_valid)} → {len(X_valid_t)} | test {len(X_test)} → {len(X_test_t)}")
assert len(X_valid_t) == len(X_valid) and len(X_test_t) == len(X_test), \
    "valid/test 행이 줄었음 — 날짜 정합 확인 필요!"

# ==========================================================================
# 4. baseline 기준 — ★ 0.5 아님. train 다수 클래스를 항상 찍는 모델의 정확도
#    (사전적 기준: 학습 시점에 알 수 있는 정보만 사용)
#    현재 데이터에서는 valid 0.5409 / test 0.5568 이 나옵니다.
# ==========================================================================
maj = int(round(y_train_t.mean()))
BASE_ACC = {"valid": float((y_valid_t == maj).mean()),
            "test":  float((y_test_t == maj).mean())}
print("다수 클래스 baseline:", {k: round(v, 4) for k, v in BASE_ACC.items()})

# ==========================================================================
# 5. 실험 정의 ①~⑥
#    ⑤ 변동성만 : TDA norm 이 기존 변동성 피처와 같은 걸 재는지 가르는 대조군
#    ⑥ 저중복 TDA : TDA 8개 중 Volatility_20 과 |상관| 0.5 미만인 5개만
#       (소현 train 기준 — L1_H0 0.81 / L2_H0 0.81 / L1_H1_std20 0.64 는 제외)
#       "④가 ⑤보다 낮은 건 중복 피처가 신호를 희석했기 때문"이라는 가설 검증용
# ==========================================================================
vol_cols = [c for c in feat_cols if c.startswith("Volatility_")]
lowcorr_tda = ["TDA_L1_H1", "TDA_L2_H1", "TDA_PE_H1", "TDA_PE_H0", "TDA_L1_H1_diff1"]

EXPERIMENTS = {
    "① 베이스라인 (22)":    feat_cols,
    "② +교차지수 (22+12)":  feat_cols + cross_cols,
    "③ +TDA (22+8)":       feat_cols + tda_cols,
    "②+③ 전부 (22+20)":    feat_cols + cross_cols + tda_cols,
    "④ TDA만 (8)":         tda_cols,
    "④' 교차지수만 (12)":   cross_cols,
    "⑤ 변동성만 (4)":       vol_cols,          # ← ④와 짝지어 보기
    "⑥ 저중복 TDA (5)":     lowcorr_tda,       # ← ④와 짝지어 보기
}

# 사전 진단: TDA 피처가 기존 변동성과 얼마나 겹치는가
_diag = pd.concat([X_train_t[tda_cols], X_train_t[vol_cols]], axis=1).corr()
print("\nTDA × Volatility_20 상관:")
print(_diag.loc[tda_cols, "Volatility_20"].round(3).to_string())
print("→ 0.8 이상이면 그 피처는 사실상 변동성의 재탕. ⑥은 이 값이 낮은 것만 모은 실험입니다.")

# ==========================================================================
# 6. 실험 매트릭스 실행 — ★ 이 셀은 그대로 두세요 (네 명 공용)
#    make_model 은 "모델 만드는 함수를 받을 빈 자리"입니다. 실제 모델은
#    §8에서 run_matrix(xgb, ...) 처럼 넘기면 그때 채워집니다.
# ==========================================================================
from sklearn.preprocessing import StandardScaler

def run_matrix(make_model, name="model", split="valid",
               use_scaled=False, task="clf", seeds=(42,)):
    """
    make_model : 시드를 받아 모델을 돌려주는 함수. 예) lambda s: XGBClassifier(random_state=s)
                 → 함수 정의를 고치는 게 아니라, 호출할 때 넘기는 값입니다.
    split : "valid" (기본, 튜닝·비교용) / "test" (최종 1회만)
    task  : "clf" 분류 / "reg" 회귀(Ridge·Lasso — 예측수익률 부호로 방향 판정)
    seeds : 여러 개 주면 예측값 평균 (Bagging처럼 시드 변동이 큰 모델용)
    """
    Xte_t = {"valid": X_valid_t, "test": X_test_t}[split]
    yte   = {"valid": y_valid_t, "test": y_test_t}[split]
    yr_te = {"valid": yr_valid_t, "test": yr_test_t}[split]
    ytr = yr_train_t if task == "reg" else y_train_t

    rows, probs = [], {}
    for exp_name, cols in EXPERIMENTS.items():
        Xtr, Xte = X_train_t[cols], Xte_t[cols]
        if use_scaled:
            sc = StandardScaler().fit(Xtr)          # ★ train 에만 fit
            Xtr = pd.DataFrame(sc.transform(Xtr), index=Xtr.index, columns=cols)
            Xte = pd.DataFrame(sc.transform(Xte), index=Xte.index, columns=cols)

        preds, seed_accs = [], []
        for s in seeds:
            m = make_model(s)
            m.fit(Xtr, ytr)
            if task == "clf" and hasattr(m, "predict_proba"):
                q = m.predict_proba(Xte)[:, 1]
            else:
                q = m.predict(Xte)
            preds.append(q)
            thr_q = 0.5 if task == "clf" else 0.0
            seed_accs.append(accuracy_score(yte, (q > thr_q).astype(int)))
        p = np.mean(preds, axis=0)
        probs[exp_name] = p

        if task == "reg":
            res = evaluate(yte, p, name=exp_name, task=task,
                           y_true_reg=yr_te, pred_reg=p)
        else:
            res = evaluate(yte, p, name=exp_name, task=task)

        # balanced_acc — 0.5 근처면 '한쪽 클래스로만 찍는 중'이라는 뜻.
        #   임계값 기반 지표(accuracy·f1·gap_pp)가 정보를 못 담는 상황을 즉시 알려줌.
        thr_p = 0.5 if task == "clf" else 0.0
        res["balanced_acc"] = balanced_accuracy_score(
            np.asarray(yte, dtype=int), (p > thr_p).astype(int))
        res["model"] = name                      # 모델명 (실험명과 분리 → 표가 안 깨짐)
        res["n_feat"] = len(cols)
        res["split"] = split
        res["baseline_acc"] = BASE_ACC[split]
        res["gap_pp"] = (res["accuracy"] - BASE_ACC[split]) * 100
        if len(seeds) > 1:
            res["acc_std_seed"] = float(np.std(seed_accs, ddof=1))
        rows.append(res)

    df = pd.DataFrame(rows).round(4)
    front = ["model", "experiment", "n_feat", "accuracy", "baseline_acc", "gap_pp",
             "balanced_acc", "roc_auc", "f1", "n"]
    cols_order = front + [c for c in df.columns if c not in front]
    return df[cols_order], probs

# ==========================================================================
# 7. 핵심 비교의 ΔAUC 부트스트랩 CI (TDA 노트북 §8과 동일)
# ==========================================================================
def boot_auc_diff(y, p_a, p_b, n_boot=2000, seed=42):
    rng = np.random.default_rng(seed)
    y = np.asarray(y, dtype=int); p_a = np.asarray(p_a); p_b = np.asarray(p_b)
    diffs = []
    for _ in range(n_boot):
        i = rng.integers(0, len(y), len(y))
        if len(np.unique(y[i])) < 2:
            continue
        diffs.append(roc_auc_score(y[i], p_b[i]) - roc_auc_score(y[i], p_a[i]))
    lo, hi = np.percentile(diffs, [2.5, 97.5])
    return float(np.mean(diffs)), float(lo), float(hi)


COMPARISONS = [
    ("① 베이스라인 (22)",   "② +교차지수 (22+12)"),
    ("① 베이스라인 (22)",   "③ +TDA (22+8)"),
    ("② +교차지수 (22+12)", "②+③ 전부 (22+20)"),
    ("⑤ 변동성만 (4)",      "④ TDA만 (8)"),        # TDA가 변동성을 넘어서는가
    ("④ TDA만 (8)",        "⑥ 저중복 TDA (5)"),    # 희석 가설 검증
    ("⑤ 변동성만 (4)",      "⑥ 저중복 TDA (5)"),    # 저중복 TDA vs 변동성 정면 비교
]

def compare_table(probs, split="valid"):
    y = {"valid": y_valid_t, "test": y_test_t}[split]
    rows = []
    for a, b in COMPARISONS:
        m, lo, hi = boot_auc_diff(y, probs[a], probs[b])
        rows.append({"비교": f"{a} → {b}", "ΔAUC": m,
                     "95% CI 하한": lo, "95% CI 상한": hi,
                     "0 포함": "예" if lo <= 0 <= hi else "아니오"})
    print("※ CI가 0을 포함하면 '이 표본에서 차이가 확인되지 않음'이 정확한 표현.")
    return pd.DataFrame(rows).round(4)

# ==========================================================================
# 8. ★ 여기만 각자 것으로 — 아래 자기 블록의 주석만 풀면 끝
#    모델을 lambda 로 만들어 run_matrix 의 첫 인자로 넘깁니다.
#
#  ▸ 결과 읽는 법 (중요)
#    · 공통 축은 ROC-AUC 입니다. 네 모델 모두 AUC 로 비교합니다.
#    · accuracy 는 보조 지표이고, XGBoost 에서만 사용 불가입니다.
#      (트리 6그루라 예측확률이 0.53~0.56에 갇혀 전부 '상승'으로 분류 →
#       accuracy·f1·gap_pp 가 다수 클래스 값에 고정. balanced_acc 0.5가 그 신호)
#    · Ridge/Lasso 는 R² 를 함께 봅니다.
#    · 비교는 '모델 간'이 아니라 '실험 간'입니다. 자기 모델 안에서 ①vs③, ④vs⑤,
#      ④vs⑥ 을 보고, 네 사람의 판정 방향이 일치하는지를 8/22에 확인합니다.
# ==========================================================================

# ── 소진 (Bagging / GBM) ────────────────────────────────────────────────
from sklearn.ensemble import BaggingClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier

bag = lambda s: BaggingClassifier(DecisionTreeClassifier(),
                                   n_estimators=50, max_samples=0.8, random_state=s)
gbm = lambda s: GradientBoostingClassifier(n_estimators=100, learning_rate=0.1,
                                            max_depth=3, random_state=s)
SEEDS = (0, 1, 2, 3, 4, 42, 100)          # 4차 회의에서 쓰신 7개 그대로
res_bag, prob_bag = run_matrix(bag, "Bagging", seeds=SEEDS)
res_gbm, prob_gbm = run_matrix(gbm, "GBM",     seeds=SEEDS)
combined_results = pd.concat([res_bag, res_gbm], ignore_index=True)
display(combined_results)
display(compare_table(prob_bag))
display(compare_table(prob_gbm))
#
#  ▸ 시드 7개 평균으로 보는 이유: Bagging의 시드 표준편차가 0.0200인데
#    TDA 추가 효과는 커야 0.01~0.02 수준이라, 단일 실행으로는 판정 불가.
#    결과표의 acc_std_seed 컬럼을 gap_pp 와 반드시 같이 읽어 주세요.
#  ▸ baseline 은 0.5가 아니라 BASE_ACC(valid 0.5409 / test 0.5568)입니다.
#    4차 회의 표를 이 기준으로 다시 계산하면 GBM(test) 0.5160 → -4.1%p 가 됩니다.


# ── 서연 (XGBoost) ──────────────────────────────────────────────────────
# from xgboost import XGBClassifier
#
# xgb = lambda s: XGBClassifier(objective="binary:logistic",
#                               n_estimators=6, max_depth=2, learning_rate=0.03,
#                               min_child_weight=5, reg_lambda=5.0,
#                               subsample=0.8, colsample_bytree=0.8,
#                               tree_method="hist", random_state=s,
#                               n_jobs=-1, verbosity=0)
# res_xgb, prob_xgb = run_matrix(xgb, "XGBoost", seeds=(42, 202, 777, 1234, 20250815))
# display(res_xgb); display(compare_table(prob_xgb))
#
#  ▸ 전처리 재실행 불필요 — 배포된 npz를 쓰면 END_DATE 2026-01-01로 자동 통일됩니다.
#    (기존 노트북 §1~§16을 건너뛰고 이 공통 코드부터 실행)
#  ▸ 하이퍼파라미터는 ①에서 찾은 값을 ①~⑥에 그대로 고정합니다. 조합별 재탐색은
#    안 하는 쪽을 권합니다 — 1-SE 이내가 36/36이었으니 재탐색해도 같은 결론이 나오고,
#    실험마다 다른 설정을 쓰면 "TDA 효과"와 "튜닝 효과"가 섞입니다.
#    (현재 설정은 2025년 데이터로 고른 값이라는 점만 발표에 명시하면 충분합니다)
#  ▸ 여유가 되면 ③에서만 기존 TimeSeriesSplit 탐색을 한 번 더 돌려
#    1-SE 이내 조합이 36/36에서 줄어드는지 확인 → 줄면 "선택 가능한 신호가 생겼다"는 증거.


# ── 다영 (Ridge / Lasso) ────────────────────────────────────────────────
# from sklearn.linear_model import Ridge, Lasso
#
# ridge = lambda s: Ridge(alpha=1.0)
# lasso = lambda s: Lasso(alpha=1e-4, max_iter=10000)
# res_r, prob_r = run_matrix(ridge, "Ridge", use_scaled=True, task="reg")
# res_l, prob_l = run_matrix(lasso, "Lasso", use_scaled=True, task="reg")
# display(pd.concat([res_r, res_l], ignore_index=True))
#
#  ▸ task="reg" → 예측수익률의 부호로 방향을 판정하고, AUC는 예측수익률을 점수로 씁니다.
#    rmse·r2 컬럼도 같이 나오니 기존 회귀 지표 서사를 그대로 이어갈 수 있습니다.
#  ▸ Lasso 계수표를 꼭 같이 뽑아 주세요 (아래). 22개 중 20개를 0으로 만드셨는데,
#    TDA 피처가 0이 아닌 채로 살아남는지가 "선형이 못 잡는 정보가 TDA에 있는가"의
#    가장 직접적인 답입니다.
#
# sc = StandardScaler().fit(X_train_t[feat_cols + tda_cols])
# lz = Lasso(alpha=1e-4, max_iter=10000).fit(
#         sc.transform(X_train_t[feat_cols + tda_cols]), yr_train_t)
# coef = pd.Series(lz.coef_, index=feat_cols + tda_cols)
# print("0이 아닌 계수:", int((coef != 0).sum()), "/", len(coef))
# display(coef[coef != 0].sort_values(key=abs, ascending=False).round(6))


# ── 소현 (TDA / RandomForest) ───────────────────────────────────────────
# from sklearn.ensemble import RandomForestClassifier
#
# rf = lambda s: RandomForestClassifier(n_estimators=300, min_samples_leaf=20,
#                                       random_state=s, n_jobs=-1)
# res_rf, prob_rf = run_matrix(rf, "RandomForest")
# display(res_rf); display(compare_table(prob_rf))
#
#  ▸ 본 노트북 §8이 원본입니다. §8의 EXPERIMENTS 에 "⑤ 변동성만"·"⑥ 저중복 TDA",
#    COMPARISONS 에 ⑤→④ / ④→⑥ / ⑤→⑥ 을 추가해 두면 팀 전체와 표가 맞습니다.

# ==========================================================================
# 9. 제출 형식 (5차 회의 8/22)
#    - 위 결과표를 CSV로 저장해서 노션에 올려 주세요. 파일명: matrix_{이름}.csv
#    - split="valid" 결과를 기본으로 냅니다. test는 방향 확정 후 마지막에 1회만.
# ==========================================================================
combined_results.to_csv("matrix_combined_소진.csv", index=False, encoding="utf-8-sig")

npz 로드: (3654, 22) (782, 22) (783, 22) | meta: ['^GSPC', '2005-01-01', '2026-01-01', '1', 'True']
TDA 8 | 교차지수 12
evaluate 정의 완료
train 3654 → 3645 (워밍업 제거, 앞부분만)
valid 782 → 782 | test 783 → 783
다수 클래스 baseline: {'valid': 0.5409, 'test': 0.5568}

TDA × Volatility_20 상관:
TDA_L1_H0          0.813
TDA_L1_H1          0.462
TDA_L2_H0          0.805
TDA_L2_H1          0.462
TDA_PE_H0         -0.075
TDA_PE_H1         -0.223
TDA_L1_H1_diff1    0.017
TDA_L1_H1_std20    0.643
→ 0.8 이상이면 그 피처는 사실상 변동성의 재탕. ⑥은 이 값이 낮은 것만 모은 실험입니다.


,model,experiment,n_feat,accuracy,baseline_acc,gap_pp,balanced_acc,roc_auc,f1,n,split,acc_std_seed
0,Bagging,① 베이스라인 (22),22,0.5563,0.5409,1.5345,0.5452,0.5467,0.6241,782,valid,0.0067
1,Bagging,② +교차지수 (22+12),34,0.4910,0.5409,-4.9872,0.4876,0.4979,0.5296,782,valid,0.0178
2,Bagging,③ +TDA (22+8),30,0.5486,0.5409,0.7673,0.5404,0.5460,0.6056,782,valid,0.0145
3,Bagging,②+③ 전부 (22+20),42,0.5345,0.5409,-0.6394,0.5310,0.5206,0.5718,782,valid,0.0129
4,Bagging,④ TDA만 (8),8,0.4987,0.5409,-4.2199,0.4871,0.4770,0.5758,782,valid,0.0065
5,Bagging,④' 교차지수만 (12),12,0.5243,0.5409,-1.6624,0.5152,0.5067,0.5876,782,valid,0.0167
6,Bagging,⑤ 변동성만 (4),4,0.4872,0.5409,-5.3708,0.4813,0.5050,0.5386,782,valid,0.0134
7,Bagging,⑥ 저중복 TDA (5),5,0.5141,0.5409,-2.6854,0.5083,0.5168,0.5632,782,valid,0.0077
8,GBM,① 베이스라인 (22),22,0.5627,0.5409,2.1739,0.5477,0.5610,0.6438,782,valid,0.0010
9,GBM,② +교차지수 (22+12),34,0.5448,0.5409,0.3836,0.5371,0.5422,0.6000,782,valid,0.0029


※ CI가 0을 포함하면 '이 표본에서 차이가 확인되지 않음'이 정확한 표현.


,비교,ΔAUC,95% CI 하한,95% CI 상한,0 포함
0,① 베이스라인 (22) → ② +교차지수 (22+12),-0.0488,-0.0894,-0.0075,아니오
1,① 베이스라인 (22) → ③ +TDA (22+8),-0.0006,-0.0311,0.0311,예
2,② +교차지수 (22+12) → ②+③ 전부 (22+20),0.0229,-0.0082,0.0546,예
3,⑤ 변동성만 (4) → ④ TDA만 (8),-0.0265,-0.0834,0.0310,예
4,④ TDA만 (8) → ⑥ 저중복 TDA (5),0.0398,0.0080,0.0710,아니오
5,⑤ 변동성만 (4) → ⑥ 저중복 TDA (5),0.0133,-0.0469,0.0728,예


※ CI가 0을 포함하면 '이 표본에서 차이가 확인되지 않음'이 정확한 표현.


,비교,ΔAUC,95% CI 하한,95% CI 상한,0 포함
0,① 베이스라인 (22) → ② +교차지수 (22+12),-0.0198,-0.0578,0.0195,예
1,① 베이스라인 (22) → ③ +TDA (22+8),-0.0099,-0.0391,0.0221,예
2,② +교차지수 (22+12) → ②+③ 전부 (22+20),-0.0017,-0.0296,0.0262,예
3,⑤ 변동성만 (4) → ④ TDA만 (8),-0.0188,-0.0788,0.0412,예
4,④ TDA만 (8) → ⑥ 저중복 TDA (5),0.0397,0.0091,0.0725,아니오
5,⑤ 변동성만 (4) → ⑥ 저중복 TDA (5),0.0210,-0.0373,0.0806,예
